In [13]:
import torch 
import matplotlib.pyplot as plt
import random
import numpy as np
from matplotlib.colors import ListedColormap
import dill
from sparse_generalization.envs.box_world.env import BoxWorldEnv
from sparse_generalization.envs.box_world.wrappers import make_env
from minigrid.wrappers import FullyObsWrapper
import gymnasium as gym
import cv2
import matplotlib.pyplot as plt
from torchmetrics.classification.accuracy import BinaryAccuracy
gym.register('BoxWorldEnv-v1', BoxWorldEnv)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
with open('../results/bern_mha_20seeds_24_Feb_2026__06h27m.pl', 'rb') as file:
    results = dill.load(file)

In [47]:
A = torch.eye(100) 
num_rand = 1

for _ in range(4):
    I = torch.eye(100) * 4
    indices = torch.randint(low=0, high=100, size=(2*num_rand,)).reshape(2, num_rand)
    print(indices)
    A[indices[0], indices[1]] = 1
    A = A @ I
    
A.sum()

tensor([[26],
        [43]])
tensor([[19],
        [ 5]])
tensor([[59],
        [60]])
tensor([[78],
        [22]])


tensor(25940.)

In [25]:
with open('../data/box_world/balanced_dist_500.pl', 'rb') as file:
    dataset = dill.load(file)
    file.close()

In [26]:
x_train = dataset['X_train']
y_train = dataset['Y_train']
x_test_ind = dataset['X_test_ind']
y_test_ind = dataset['Y_test_ind']
x_test_ood = dataset['X_test_ood']
y_test_ood = dataset['Y_test_ood']
edges_train = dataset['edges_train']
edges_test = dataset['edges_test_ood']

In [195]:
test = edges_train[0:50]
test = test.view(50 * 1 * 4, 2, 2)
test = test[:, :, 1] * 10 + test[:, :, 0] 
test = test.view(50, 4, 2)
A = torch.zeros(50, 100, 100)
item = test[0]
A[0, item[:, 0], item[:, 1]] = 1

In [190]:
test[0]

tensor([[[16, 54],
         [11, 12],
         [51, 52],
         [27, 26]]])

In [203]:
edges_train[3]

tensor([[[[6, 7],
          [5, 8]],

         [[4, 6],
          [5, 6]],

         [[1, 7],
          [2, 7]],

         [[8, 5],
          [7, 5]]]])

In [145]:
def get_edges(obs, residual=False):
    edges = []
    # get goal-lock 
    x_goal, y_goal = torch.where(obs[:, :, 0] == 8)
    goal = torch.stack([x_goal, y_goal], dim=1).squeeze()
    edges.append(((goal[0]+1, goal[1]), (goal[0], goal[1])))
    # for each key check key + 1 in lock then a combo
    xs, ys = torch.where(x_train[0][:, :, 0] == 13)
    locks = torch.stack([xs, ys], dim=1)
    locks = locks[~(locks == torch.tensor([goal[0]+1, goal[1]])).all(dim=1)]
    
    xs, ys = torch.where(x_train[0][:, :, 0] == 12)
    keys = torch.stack([xs, ys], dim=1)
    for lock in locks:
        key = (lock[0]-1, lock[1])
        edges.append(((lock[0], lock[1]), (key)))
        keys = keys[~(keys == torch.tensor([lock[0]-1, lock[1]])).all(dim=1)]
        
    # remaining key is residual or looking from agent
    first_key = keys.squeeze()
    if not residual: 
        xs, ys = torch.where(x_train[0][:, :, 0] == 10)
        agent = torch.stack([xs, ys], dim=1).squeeze()
        edges.append(((agent[0], agent[1]), (first_key[0], first_key[1])))
    
    return edges

In [43]:
from sparse_generalization.layers.oracle import MultiHeadAttentionOracle

mha = MultiHeadAttentionOracle(embed_size=3, num_heads=3)
x_train = x_train.view(500, 100, 3).float()
mha(x_train, x_train, x_train, edges_train)[1].shape

torch.Size([500, 100, 100])

In [ ]:
# %% [markdown]
# # SPARTAN Model Training and Sparsity Analysis Notebook

# %%
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# Import your SPARTAN class directly from your script file
from sparse_generalization.models.spartan import SPARTAN

import random
import numpy as np

def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if you use multi-GPU
    
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Execute the seed setting
seed_everything(0)

# Ensure graphics render inline
%matplotlib inline

# %% [markdown]
# ## 1. Synthetic Dataset Generation (Strict Permutations, Pool Size = 16)

# %%
device = "cuda" if torch.cuda.is_available() else "cpu"

class Logger:
    def __init__(self):
        pass

    def log_metrics(self, a, step):
        pass

# Dimensions setup - Strictly bounded pool
width, height, channels = 4, 4, 1
num_embeddings = width * height  # Exactly 16 unique tokens available (0 to 15)
num_samples = 5000  

X_list = []
Y_list = []

half_samples = num_samples // 2

def check_row_rule(grid):
    """Helper to verify if a 4x4 grid meets the Class 1 criteria."""
    for row_idx in range(width):
        row = grid[row_idx]
        if (0 in row and 1 in row) or (8 in row and 9 in row):
            return True
    return False

# --- 1. Generate Class 0 Samples (No rules allowed, strict permutations of 0-15) ---
print("Generating Class 0 samples...")
while len(X_list) < half_samples:
    # Generate a perfect permutation of numbers 0 to 15
    grid = torch.randperm(num_embeddings).float().view(width, height)
    
    # Discard if it accidentally satisfies the row rule
    if not check_row_rule(grid):
        X_list.append(grid.unsqueeze(-1))
        Y_list.append(torch.tensor([0.0]))

# --- 2. Generate Class 1 Samples (Forced rules, strict permutations of 0-15) ---
print("Generating Class 1 samples...")
while len(X_list) < num_samples:
    # 1. Pick which rule pair to force inject
    target_pair = torch.rand(1).item()
    injected_tokens = [0.0, 1.0] if target_pair > 0.5 else [8.0, 9.0]
    
    # 2. Get the remaining 14 tokens in the 0-15 range to maintain unique cells
    forbidden = set(injected_tokens)
    remaining_pool = torch.tensor([float(x) for x in range(num_embeddings) if x not in forbidden], dtype=torch.float32)
    
    # Shuffle the background tokens
    background_tokens = remaining_pool[torch.randperm(len(remaining_pool))]
    
    # 3. Create a flat grid array
    flat_grid = torch.zeros(num_embeddings)
    
    # Randomly pick a target row, and 2 unique column positions within that row
    target_row = torch.randint(0, width, (1,)).item()
    col_indices = torch.randperm(height)[:2]
    
    # Map 2D coordinates to flat 1D indices
    idx1 = target_row * height + col_indices[0]
    idx2 = target_row * height + col_indices[1]
    
    # Inject the rule pair
    flat_grid[idx1] = injected_tokens[0]
    flat_grid[idx2] = injected_tokens[1]
    
    # Fill the remaining 14 slots with the rest of the 0-15 numbers
    bg_idx = 0
    for i in range(num_embeddings):
        if i != idx1 and i != idx2:
            flat_grid[i] = background_tokens[bg_idx]
            bg_idx += 1
            
    # Reshape back to 4x4 matrix
    grid = flat_grid.view(width, height)
    
    X_list.append(grid.unsqueeze(-1))
    Y_list.append(torch.tensor([1.0]))

# Stack and consolidate
X_data = torch.stack(X_list)
Y_data = torch.stack(Y_list)

X_data = torch.nn.functional.one_hot(X_data.long(), num_classes=num_embeddings).squeeze().float()

# Shuffle the final collection to mix classes
shuffle_idx = torch.randperm(num_samples)
X_data = X_data[shuffle_idx]
Y_data = Y_data[shuffle_idx]

dataset = TensorDataset(X_data, Y_data)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

print(f"\nDataset generated successfully with num_embeddings={num_embeddings}!")
print(f"X shape: {X_data.shape} | Y shape: {Y_data.shape}")
print(f"Verify unique values per grid sample: {len(torch.unique(X_data[0]))} values (should be 16)")


# %% [markdown]
# ## 2. Instantiating the SPARTAN Object
# %%
logger = Logger()

model = SPARTAN(
    inp_dim=num_embeddings,
    out_dim=1,
    model_dim=32,
    alpha=0.1, 
    num_layers=1,
    num_heads=1,
    include_sparsity=False,   # Standard sparsity enforcement
    lagrangian=False,         
    embedding_inp=False,
    separate_mask=True, 
    agg_pool=True,            # Output pool collapses sequence dim
    num_embeddings=num_embeddings,
    device=device,
    logger=logger
).to(device)

print("SPARTAN model successfully instantiated and pushed to device.")

# %% [markdown]
# ## 3. Model Training
# %%
epochs = 2000
print(f"Starting training loop for {epochs} epochs...")

# Unpacking the 9 elements returned by your fit method
losses, accs, sparses, _, _, _, _, _, _ = model.fit(dataloader, num_epochs=epochs, testloaders=[])
print(f"Model max paths: {model.max_paths}")

# %% [markdown]
# ## 4. Extracting Internal Attention and Sparsity States
# %%
model.eval()
sample_x, _ = next(iter(dataloader))
sample_x = sample_x[:1].to(device) # Isolate a single sample

with torch.no_grad():
    # Unpacking all 5 outputs returned by your latest forward pass definition
    out, final_path, mask_attns, attns, masks = model(sample_x)

print("--- Matrix Shape Summary ---")
for i in range(len(attns)):
    is_agg = (i == len(attns) - 1 and model.agg_pool)
    name = f"Layer {i+1}" if not is_agg else "Aggregation Layer"
    print(f"{name} -> Attn: {list(attns[i].shape)} | Mask Attn: {list(mask_attns[i].shape)} | Mask: {list(masks[i].shape)}")

print(f"Final Compiled Path Mask Shape: {list(final_path.shape)}\n")

# %% [markdown]
# ## 5. Visualizing Attentions, Masked Attentions, and Masks Layer-by-Layer
# This handles explicit vector squeezing for (1, 1, 16) or (1, 16) dimensions in the aggregation tier.

# %%
num_plots = len(attns)
# 3 Columns: [Attention] | [Masked Attention] | [Extracted Mask]
fig, axes = plt.subplots(num_plots, 3, figsize=(18, 4.0 * num_plots))

# Handle single-layer array boundary edge cases for matplotlib stability
if num_plots == 1:
    axes = [axes]

for idx in range(num_plots):
    is_agg = (idx == num_plots - 1 and model.agg_pool)
    layer_name = f"Layer {idx+1}" if not is_agg else "Aggregation Pool"
    
    # Isolate first item in the batch and completely squeeze out extra dummy/head dims
    # Normal layer: (B=1, H=1, 16, 16) -> (16, 16)
    # Agg layer with explicit heads: (B=1, H=1, 1, 16) or (B=1, 1, 16) -> (1, 16)
    attn_img = attns[idx][0].squeeze().cpu().numpy()
    mask_attn_img = mask_attns[idx][0].squeeze().cpu().numpy()
    mask_img = masks[idx][0].squeeze().cpu().numpy()
    
    # If the vector squished to a purely 1D numpy array shape like (16,), expand to (1, 16) for imshow
    if attn_img.ndim == 1:
        attn_img = attn_img[None, :]
        mask_attn_img = mask_attn_img[None, :]
        mask_img = mask_img[None, :]

    # Dynamically pick display mode for 2D vs 1D tracking rows
    is_flat_shape = attn_img.shape[0] == 1
    aspect_mode = 'auto' if is_flat_shape else 'equal'
    interp_mode = 'nearest' if is_flat_shape else None

    # -------------------------------------
    # Column 1: Raw Attention Map
    # -------------------------------------
    ax_attn = axes[idx][0]
    im1 = ax_attn.imshow(attn_img, cmap='viridis', aspect=aspect_mode, interpolation=interp_mode)
    ax_attn.set_title(f"{layer_name}\nAttention Matrix {attn_img.shape}")
    fig.colorbar(im1, ax=ax_attn, fraction=0.046, pad=0.04)
    
    # -------------------------------------
    # Column 2: Masked Attention Map
    # -------------------------------------
    ax_m_attn = axes[idx][1]
    im2 = ax_m_attn.imshow(mask_attn_img, cmap='plasma', aspect=aspect_mode, interpolation=interp_mode)
    ax_m_attn.set_title(f"{layer_name}\nMasked Attention {mask_attn_img.shape}")
    fig.colorbar(im2, ax=ax_m_attn, fraction=0.046, pad=0.04)
    
    # -------------------------------------
    # Column 3: Sparsity Selection Mask
    # -------------------------------------
    ax_mask = axes[idx][2]
    im3 = ax_mask.imshow(mask_img, cmap='inferno', aspect=aspect_mode, interpolation=interp_mode)
    ax_mask.set_title(f"{layer_name}\nExtracted Mask {mask_img.shape}")
    fig.colorbar(im3, ax=ax_mask, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# %% [markdown]
# ## 6. Softmax Axis and Sum Verification

# %%
print("=== Softmax Sum Evaluation (Tolerance = 1e-4) ===")

for idx, attn_tensor in enumerate(attns):
    is_agg = (idx == len(attns) - 1 and model.agg_pool)
    layer_name = f"Layer {idx+1}" if not is_agg else "Aggregation Pool"
    
    # Target element 0 and strip out batch/head dims for numerical profiling
    matrix = attn_tensor[0].squeeze()
    if matrix.ndim == 1:
        matrix = matrix[None, :] # map to shape (1, 16)
        
    row_sums = matrix.sum(dim=-1)
    col_sums = matrix.sum(dim=-2)
    
    rows_match = torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4)
    cols_match = torch.allclose(col_sums, torch.ones_like(col_sums), atol=1e-4)
    
    print(f"\n[{layer_name}] Matrix Shape: {list(matrix.shape)}")
    print(f"  -> Rows sum to 1.0? {rows_match} (Min sum: {row_sums.min().item():.4f}, Max sum: {row_sums.max().item():.4f})")
    print(f"  -> Cols sum to 1.0? {cols_match} (Min sum: {col_sums.min().item():.4f}, Max sum: {col_sums.max().item():.4f})")

In [12]:
attention_logits = torch.randn((1, 16, 16))

probs = torch.sigmoid(attention_logits + 0.5)
logits = torch.log(probs.unsqueeze(-1))
z = torch.log(1.0-probs.detach().unsqueeze(-1))
logits = torch.concat((logits,z), dim=-1)
A = torch.nn.functional.gumbel_softmax(logits, tau=1.0, hard=True)[:, :, :, 0] * 1.0

edges_logit = attention_logits.view(1, -1)  # (b*h, l*l)
edges_logit = torch.stack(
    [torch.zeros_like(edges_logit), edges_logit + 0.5], dim=-1
)
A2 = torch.nn.functional.gumbel_softmax(edges_logit, tau=1.0, hard=True)
A2 = A2[:, :, -1].reshape(1, 16, 16)

In [13]:
A

tensor([[[1., 0., 1., 1., 0., 1., 1., 0., 0., 1., 0., 0., 0., 1., 1., 0.],
         [0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 1., 1., 1., 1., 0., 0.],
         [1., 0., 0., 1., 1., 0., 1., 1., 1., 0., 1., 0., 1., 0., 0., 1.],
         [0., 1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 1., 0., 1., 1., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 0., 0., 1., 0., 0., 1.],
         [1., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 1., 0., 1., 0.],
         [1., 0., 1., 0., 0., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 0., 1., 0., 1., 1., 0., 1., 0., 1., 0., 0.],
         [1., 1., 1., 0., 0., 0., 0., 0., 0., 1., 0., 1., 1., 0., 1., 0.],
         [1., 1., 1., 1., 1., 0., 1., 1., 0., 1., 0., 0., 1., 1., 0., 0.],
         [1., 0., 1., 1., 0., 1., 0., 1., 1., 0., 0., 1., 1., 0., 1., 1.],
         [0., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 0., 0., 1.],
         [0., 1., 1., 0., 1., 1., 1., 0., 1., 1., 1., 1., 0., 0., 0., 0.],
         [0., 0., 1., 0.,

In [18]:
paths = torch.ones((16, 16)) * 1 + torch.eye(16)
test = torch.zeros((1, 16))
test[:, -1] = 1
test[:, -2] = 1
(test @ paths).sum()

tensor(34.)

In [16]:
paths = torch.ones((16, 16)) * 1 + torch.eye(16)
for l in range(0):
    multiplier = torch.ones((16, 16)) + torch.eye( 16)
    paths = paths @ multiplier

multiplier = torch.ones((1, 16)) * 1
paths = multiplier @ paths
paths.sum()

tensor(272.)